# Validate Q2 2024's channel report

*Data Analysis in Python · Topic 14.3*

Your team's Q2 2024 channel report goes out every quarter, and from now on, it should check itself first. Write a validation function that checks the report against the sessions table, make it stop the notebook when a check fails, and run it on an AI-written version of the report.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the two cells that build the `sessions` table, and the two cells that create the `report` and `ai_report` tables.
2. In the first cell marked ✏️, write `validate_report(sessions, report, start, end)`: a function that returns a dictionary of named checks, each `True` or `False`:
   - every day from `start` to `end` is in the `sessions` table
   - the report's sessions add up to the `sessions` table's
   - the report's revenue adds up to the `sessions` table's, to the cent
   - each conversion rate in the report is its transactions divided by its sessions
3. In the second cell marked ✏️, create `checks`: the checks for the `sessions` and `report` tables. Then `assert` that every check passes, with `checks` as the message.
4. In the third cell marked ✏️, create `ai_checks`: the checks for the `sessions` and `ai_report` tables. Don't `assert` them: look at which ones fail.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `validate_report(sessions, report, START, END)` returns four checks, all `True`, and the `assert` lets the notebook go on. The `ai_checks` dictionary has two `False` checks: the revenue and the rates.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2024-04", "2024-05", "2024-06"]
START, END = "2024-04-01", "2024-06-30"  # Q2 2024

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}
raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Channel groups, with topic 4.2's steps and Foundations topic 12.3's function. Run it, and leave it as it is.
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium, with Foundations topic 12.3's rules."""
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"


def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])


sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions["channel_group"].value_counts()

## The two reports

In [ ]:
# The team's report, as in topic 14.1. Run it, and leave it as it is.
report = sessions.groupby("channel_group", as_index=False)[["sessions", "transactions", "revenue"]].sum()
report["conversion_rate"] = report["transactions"] / report["sessions"]
report = report.sort_values("revenue", ascending=False).reset_index(drop=True)
report

In [ ]:
# An AI-written report, as in topic 14.2. Run it, and leave it as it is.
valid = sessions[sessions["revenue"] >= 0]  # Drop rows with negative revenue, which are invalid values
daily = valid.groupby(["channel_group", "date"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
daily["conversion_rate"] = daily["transactions"] / daily["sessions"]
ai_report = daily.groupby("channel_group", as_index=False).agg(
    sessions=("sessions", "sum"),
    transactions=("transactions", "sum"),
    revenue=("revenue", "sum"),
    conversion_rate=("conversion_rate", "mean"),
)
ai_report = ai_report.sort_values("revenue", ascending=False).reset_index(drop=True)
ai_report

## The validation function

In [ ]:
def validate_report(sessions, report, start, end):
    """Return each check's name and whether the report passes it."""
    return {
        "every day is there": sessions["date"].nunique() == len(pd.date_range(start, end)),
        "sessions add up": bool(report["sessions"].sum() == sessions["sessions"].sum()),
        "revenue adds up": bool(round(report["revenue"].sum(), 2) == round(sessions["revenue"].sum(), 2)),
        "rates from totals": bool(((report["conversion_rate"] - report["transactions"] / report["sessions"]).abs() < 0.000001).all()),
    }

## Validate the report

In [ ]:
checks = validate_report(sessions, report, START, END)
assert all(checks.values()), checks
checks

## Validate the AI's report

In [ ]:
ai_checks = validate_report(sessions, ai_report, START, END)
ai_checks

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
    message = f"{type(error).__name__}: {error}" if error is not None else ""
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    elif checked_before:
        results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
    elif [source for source in history if source.strip() and history.count(source) > 1]:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        first_line = repeated[0].strip().splitlines()[0]
        results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
    elif "AssertionError" in message:
        results.append((None, f"An assert stopped the notebook: {error}. If it's the one on ai_checks, remove it: the AI's report should fail, and the notebook should go on. Then choose Runtime → Restart session and run all."))
    elif error is not None:
        results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
    else:
        results.append((True, "Every cell in this session ran once, with no errors."))

    validate_report = found.get("validate_report")
    sessions, report = found.get("sessions"), found.get("report")
    start, end = "2024-04-01", "2024-06-30"

    def rebuilt(table):
        """Return the team's report, built from another sessions table."""
        new = table.groupby("channel_group", as_index=False)[["sessions", "transactions", "revenue"]].sum()
        new["conversion_rate"] = new["transactions"] / new["sessions"]
        return new

    def passes(table, a_report):
        try:
            result = validate_report(table, a_report, start, end)
        except Exception as problem:
            return f"{type(problem).__name__}: {problem}"
        return isinstance(result, dict) and all(result.values())

    if not callable(validate_report):
        results.append((False, "validate_report should be a function: def validate_report(sessions, report, start, end): ..."))
    elif not isinstance(sessions, pd.DataFrame) or not isinstance(report, pd.DataFrame):
        results.append((False, "Run the cells that create the sessions and report tables, and leave them as they are."))
    else:
        good = validate_report(sessions, report, start, end)
        short = sessions[~sessions["date"].between("2024-06-10", "2024-06-16")]
        no_refunds = rebuilt(sessions[sessions["revenue"] >= 0])
        no_display = report[report["channel_group"] != "Display"]
        mean_rates = report.copy()
        mean_rates["conversion_rate"] = mean_rates["conversion_rate"] * 1.02
        if not isinstance(good, dict) or not good:
            results.append((False, f"validate_report() should return a dictionary of named checks, each True or False. It returns {good!r}."))
        elif not all(good.values()):
            results.append((False, f"validate_report() should pass on the team's report, and it doesn't: {good}. Check the failing check's code."))
        elif passes(short, rebuilt(short)) is True:
            results.append((False, "validate_report() passes when June 10 to 16 are missing from the sessions table, and the report is built from what's left. Check that every day from start to end is there: pd.date_range(start, end)."))
        elif passes(sessions, no_display) is True:
            results.append((False, "validate_report() passes a report that's missing the Display row. Check that the report's sessions add up to the sessions table's."))
        elif passes(sessions, no_refunds) is True:
            results.append((False, "validate_report() passes a report without the refund rows. Check that the report's revenue adds up to the sessions table's, to the cent."))
        elif passes(sessions, mean_rates) is True:
            results.append((False, "validate_report() passes conversion rates that are 2% off their totals. Check each rate against transactions / sessions."))
        else:
            results.append((True, f"validate_report() passes on the team's report with {len(good)} checks, and fails on missing days, missing rows, missing refunds and wrong rates."))

    checks = found.get("checks")
    asserted = isinstance(inputs, list) and any("assert" in source and "checks" in source for source in inputs[1:-1])
    if not isinstance(checks, dict) or not checks:
        results.append((False, "checks should be validate_report(sessions, report, START, END)."))
    elif not all(checks.values()):
        results.append((False, f"Every check should pass for the team's report: {checks}."))
    elif not asserted:
        results.append((False, "Make a failure stop the notebook: assert all(checks.values()), checks."))
    else:
        results.append((True, "Every check passes for the team's report, and an assert stops the notebook if one doesn't."))

    ai_checks = found.get("ai_checks")
    if not isinstance(ai_checks, dict) or not ai_checks:
        results.append((False, "ai_checks should be validate_report(sessions, ai_report, START, END)."))
    elif all(ai_checks.values()):
        results.append((False, f"Every check passes for the AI's report: {ai_checks}. Its revenue and its rates are wrong, so two checks should fail."))
    else:
        failed = [name for name, passed in ai_checks.items() if not passed]
        results.append((True, f"The AI's report fails {len(failed)} checks: {', '.join(failed)}."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
def validate_report(sessions, report, start, end):
    """Return each check's name and whether the report passes it."""
    return {
        "every day is there": sessions["date"].nunique() == len(pd.date_range(start, end)),
        "sessions add up": bool(report["sessions"].sum() == sessions["sessions"].sum()),
        "revenue adds up": bool(round(report["revenue"].sum(), 2) == round(sessions["revenue"].sum(), 2)),
        "rates from totals": bool(((report["conversion_rate"] - report["transactions"] / report["sessions"]).abs() < 0.000001).all()),
    }
```

```python
checks = validate_report(sessions, report, START, END)
assert all(checks.values()), checks
checks
```

```python
ai_checks = validate_report(sessions, ai_report, START, END)
ai_checks
```

</details>